02_feature_engineering_modeling.ipynb

Step 9 to Step 14

### STEP 9 : Advanced Cleaning

In this step, we refine the dataset by selecting only meaningful features, handling missing values, and preparing the dataset for modeling.

We focus on retaining high-signal features across behavioral, customer profile, payment, time, and scoring dimensions while removing unnecessary or redundant columns.

1. Feature Selection

We retain only the most important features identified from EDA and hypothesis testing.

These features capture:
- Customer behavior
- Engagement level
- Customer profile
- Payment patterns
- Time-based trends
- Business scoring metrics

In [62]:
import pandas as pd

df = pd.read_csv("modeling_dataset.csv")

C:\Users\nehas\AppData\Local\Temp\ipykernel_22136\1053631762.py:3: DtypeWarning: Columns (0: Discount_Amount, 1: Proforma_Auto_Renewal, 2: Proforma_World_Pay_Token, 3: Current_Anchor_List, 4: Last_Renewal, 5: Last_Band) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("modeling_dataset.csv")


In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 113894 entries, 0 to 113893
Data columns (total 73 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Co_Ref                      113894 non-null  str    
 1   Renewal_Month               113894 non-null  str    
 2   Connection_Net              6392 non-null    float64
 3   Connection_Qty              6392 non-null    float64
 4   Discount_Amount             13189 non-null   str    
 5   Sustainability_Score        113894 non-null  float64
 6   Total_Renewal_Score_New     113894 non-null  float64
 7   Starting_Connection_Net     6837 non-null    float64
 8   Starting_Connection_Qty     6837 non-null    float64
 9   Last_Years_Price            105077 non-null  float64
 10  Auto_Renewal_Score          113894 non-null  int64  
 11  Status_Scores               113894 non-null  int64  
 12  Anchoring_Score             113894 non-null  float64
 13  Tenure_Scores            

In [64]:
selected_cols = ['Co_Ref',
    'total_interactions', 'cc_call_ratio', 'low_engagement', 'total_emails',
    'tenure_bucket', 'Band', 'Anchor_Group',
    'Payment_Method', 'Payment_Timeframe',
    'Renewal_Year',
    'Sustainability_Score', 'Total_Renewal_Score_New',
    'Auto_Renewal_Score', 'Anchoring_Score', 'Tenure_Scores',
    'churn',
    'Prospect_Renewal_Date', 'Closed_Date'   # needed for calculation
]

df = df[selected_cols]

In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 113894 entries, 0 to 113893
Data columns (total 19 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Co_Ref                   113894 non-null  str    
 1   total_interactions       113894 non-null  float64
 2   cc_call_ratio            113894 non-null  float64
 3   low_engagement           113894 non-null  int64  
 4   total_emails             113894 non-null  float64
 5   tenure_bucket            112740 non-null  str    
 6   Band                     113874 non-null  str    
 7   Anchor_Group             113772 non-null  str    
 8   Payment_Method           113894 non-null  str    
 9   Payment_Timeframe        101226 non-null  float64
 10  Renewal_Year             113894 non-null  int64  
 11  Sustainability_Score     113894 non-null  float64
 12  Total_Renewal_Score_New  113894 non-null  float64
 13  Auto_Renewal_Score       113894 non-null  int64  
 14  Anchoring_Score

2. Handling Payment_Timeframe Missing Values

Payment_Timeframe represents the number of days between payment and renewal.

For missing values, we derive it using:
Payment_Timeframe = Prospect_Renewal_Date - Closed_Date

This ensures business-consistent imputation.

In [66]:
# Prospect_Renewal_Date → YYYY-MM-DD
df['Prospect_Renewal_Date'] = pd.to_datetime(
    df['Prospect_Renewal_Date'],
    format='%Y-%m-%d',
    errors='coerce'
)

# Closed_Date → DD-MM-YYYY
df['Closed_Date'] = pd.to_datetime(
    df['Closed_Date'],
    format='%d-%m-%Y',
    errors='coerce'
)
# Now apply calculation
mask = df['Payment_Timeframe'].isnull()

df.loc[mask, 'Payment_Timeframe'] = (
    df.loc[mask, 'Prospect_Renewal_Date'] - df.loc[mask, 'Closed_Date']
).dt.days

In [67]:
df.isnull().sum()

Co_Ref                        0
total_interactions            0
cc_call_ratio                 0
low_engagement                0
total_emails                  0
tenure_bucket              1154
Band                         20
Anchor_Group                122
Payment_Method                0
Payment_Timeframe             0
Renewal_Year                  0
Sustainability_Score          0
Total_Renewal_Score_New       0
Auto_Renewal_Score            0
Anchoring_Score               0
Tenure_Scores                 0
churn                         0
Prospect_Renewal_Date         0
Closed_Date                   0
dtype: int64

In [68]:
df[['Co_Ref', 'Prospect_Renewal_Date', 'Closed_Date', 'Payment_Timeframe']].isnull().sum()

Co_Ref                   0
Prospect_Renewal_Date    0
Closed_Date              0
Payment_Timeframe        0
dtype: int64

3. Drop Temporary Columns

After deriving Payment_Timeframe, date columns are no longer needed.

In [69]:
df = df.drop(columns=['Prospect_Renewal_Date', 'Closed_Date'])

In [70]:
df.columns

Index(['Co_Ref', 'total_interactions', 'cc_call_ratio', 'low_engagement',
       'total_emails', 'tenure_bucket', 'Band', 'Anchor_Group',
       'Payment_Method', 'Payment_Timeframe', 'Renewal_Year',
       'Sustainability_Score', 'Total_Renewal_Score_New', 'Auto_Renewal_Score',
       'Anchoring_Score', 'Tenure_Scores', 'churn'],
      dtype='str')

5. Handling Categorical Missing Values

Missing categorical values are filled with 'Unknown'.

This preserves missingness as a meaningful signal instead of removing data.

In [71]:
df['tenure_bucket'] = df['tenure_bucket'].fillna('Unknown')
df['Band'] = df['Band'].fillna('Unknown')
df['Anchor_Group'] = df['Anchor_Group'].fillna('Unknown')


In [72]:
df.isnull().sum()

Co_Ref                     0
total_interactions         0
cc_call_ratio              0
low_engagement             0
total_emails               0
tenure_bucket              0
Band                       0
Anchor_Group               0
Payment_Method             0
Payment_Timeframe          0
Renewal_Year               0
Sustainability_Score       0
Total_Renewal_Score_New    0
Auto_Renewal_Score         0
Anchoring_Score            0
Tenure_Scores              0
churn                      0
dtype: int64

In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 113894 entries, 0 to 113893
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Co_Ref                   113894 non-null  str    
 1   total_interactions       113894 non-null  float64
 2   cc_call_ratio            113894 non-null  float64
 3   low_engagement           113894 non-null  int64  
 4   total_emails             113894 non-null  float64
 5   tenure_bucket            113894 non-null  str    
 6   Band                     113894 non-null  str    
 7   Anchor_Group             113894 non-null  str    
 8   Payment_Method           113894 non-null  str    
 9   Payment_Timeframe        113894 non-null  float64
 10  Renewal_Year             113894 non-null  int64  
 11  Sustainability_Score     113894 non-null  float64
 12  Total_Renewal_Score_New  113894 non-null  float64
 13  Auto_Renewal_Score       113894 non-null  int64  
 14  Anchoring_Score

In [74]:
df.head(10)

,Co_Ref,total_interactions,cc_call_ratio,low_engagement,total_emails,tenure_bucket,Band,Anchor_Group,Payment_Method,Payment_Timeframe,Renewal_Year,Sustainability_Score,Total_Renewal_Score_New,Auto_Renewal_Score,Anchoring_Score,Tenure_Scores,churn
0,vt6174,1.0,0.000000,0,1.0,1-3,band c1,1,card,0.0,2024,8.0,42.5,9,7.5,9.0,0
1,vd3828,1.0,0.000000,0,1.0,0-1,band c1,1,card,0.0,2025,8.0,41.5,9,7.5,8.0,0
2,dv8120,0.0,0.000000,1,0.0,5-10,band c1,1,card,0.0,2025,8.0,33.0,8,7.5,9.5,0
3,ez9894,7.0,0.625000,0,2.0,10+,band c1,1,card,0.0,2025,9.5,44.5,9,7.5,9.5,0
4,fa8957,6.0,0.285714,0,4.0,1-3,band c1,1,card,0.0,2025,9.5,42.5,9,7.5,8.5,0
5,qs2598,1.0,0.000000,0,1.0,3-5,band c1,1,card,0.0,2025,8.0,40.5,9,7.5,9.0,0
6,cj9355,2.0,0.000000,0,2.0,10+,band c1,1,card,0.0,2025,8.0,43.0,9,7.5,9.5,0
7,fp1608,2.0,0.000000,0,2.0,5-10,band c1,1,card,0.0,2024,9.5,44.5,9,7.5,9.5,0
8,tz9717,1.0,0.000000,0,1.0,5-10,band c1,1,card,0.0,2024,8.0,41.0,9,7.5,9.5,0
9,up7099,3.0,0.250000,0,2.0,1-3,band c1,1,card,0.0,2025,8.0,42.0,9,7.5,8.5,0


6. Handling Datatypes

In [75]:
cat_cols = ['tenure_bucket', 'Band', 'Anchor_Group', 'Payment_Method']

for col in cat_cols:
    df[col] = df[col].astype('category')

df['Renewal_Year'] = df['Renewal_Year'].astype('category')

In [76]:
df.dtypes

Co_Ref                          str
total_interactions          float64
cc_call_ratio               float64
low_engagement                int64
total_emails                float64
tenure_bucket              category
Band                       category
Anchor_Group               category
Payment_Method             category
Payment_Timeframe           float64
Renewal_Year               category
Sustainability_Score        float64
Total_Renewal_Score_New     float64
Auto_Renewal_Score            int64
Anchoring_Score             float64
Tenure_Scores               float64
churn                         int64
dtype: object

In [77]:
df.to_csv('step9_clean_dataset.csv', index=False)

### 10. Advanced Feature Engineering

In this step, we extract behavioral signals from:

- Renewal Calls → customer intent, complaints, churn signals
- CC Calls → issues, dissatisfaction, sentiment
- Emails → engagement, complaints, financial stress

These features help the model understand customer behavior before churn.

In [78]:
renewal_calls_clean= pd.read_csv("renewal_calls_clean.csv")
cc_calls_clean = pd.read_csv("cc_calls_clean.csv")
emails_clean = pd.read_csv("emails_clean.csv")

C:\Users\nehas\AppData\Local\Temp\ipykernel_22136\2102415877.py:1: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Value_Comparison, 20: Competitor_Benefits_Mentioned, 21: Topic_Introduced_By, 22: Percentage_Price_Increase_Mentioned, 23: Monetary_Price_Increase_Mentioned, 24: Price_Range_Mentioned, 25: Customer_Asked_For_Justification, 26: Customer_Response, 27: Desire_To_Cancel, 28: Discount_Offered, 29: Justification_Categor

In [79]:
renewal_calls_clean['Call_Date'] = pd.to_datetime(renewal_calls_clean['Call_Date'], errors='coerce')
cc_calls_clean['Call_Date'] = pd.to_datetime(cc_calls_clean['Call_Date'], errors='coerce')

In [80]:
renewal_calls_clean.columns

Index(['Call_ID', 'Call_Direction', 'Co_Ref', 'Call_Date', 'Churn_Category',
       'Complaint_Category', 'Customer_Reaction_Category',
       'Agent_Renewal_Pitch_Category', 'Customer_Renewal_Response_Category',
       'Agent_Response_Category', 'Membership_Renewal_Decision',
       'Serious_Complaint', 'Other_Complaint', 'Discussion_on_Price_Increase',
       'Renewal_Impact_Due_to_Price_Increase', 'Discount_or_Waiver_Requested',
       'Call_Reschedule_Request', 'Agent_Flagged_Membership_Status_Alert',
       'Agent_Renewal_Initiation', 'Explicit_Competitor_Mention',
       'Explicit_Switching_Intent', 'Mentioned_Competitors',
       'Price_Switching_Mentioned', 'Competitor_Value_Comparison',
       'Competitor_Benefits_Mentioned', 'Topic_Introduced_By',
       'Percentage_Price_Increase_Mentioned',
       'Monetary_Price_Increase_Mentioned', 'Price_Range_Mentioned',
       'Customer_Asked_For_Justification', 'Customer_Response',
       'Desire_To_Cancel', 'Discount_Offered', 'Justi

Important Columns:

Customer_Response
Desire_To_Cancel
Explicit_Switching_Intent
Complaint_Category

In [81]:
cc_calls_clean.columns

Index(['Contact_ID', 'Call_Date', 'Direction', 'cc_care_package',
       'cc_care_package_discussed', 'cc_urgency_getting_on_site',
       'cc_external_consultant', 'cc_agent_cross_sell_attempt',
       'cc_customer_issues_concerns',
       'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
       'cc_questionnaire_completion', 'cc_chasing_response',
       'cc_issues_within_questionnaire', 'cc_login_issues',
       'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
       'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
       'cc_dissatisfaction_support', 'cc_contractor_sentiment',
       'cc_contractor_sentiment_start_score',
       'cc_contractor_sentiment_end_score',
       'cc_contractor_sentiment_overall_score',
       'cc_contractor_sentiment_issues_score', 'cc_pricing_mentioned',
       'cc_pricing_sentiment_impact', 'cc_refund_discussed',
       'cc_contractor_suggest_leave', 'cc_contractor_complained', 'Co_Ref',
       'Analys

Important Columns:

cc_contractor_complained, cc_business_struggles_financial_hardship, cc_pricing_mentioned, cc_platform_issues, cc_dissatisfaction_support 

In [82]:
emails_clean.columns

Index(['Co_Ref', 'Time_to_Renewal', 'crm_accreditation_completed',
       'crm_timely_completion', 'crm_progress_towards_accreditation',
       'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
       'crm_contractor_engagement', 'crm_contractor_sentiment',
       'crm_contractor_sentiment_score', 'crm_dts_or_ssip_mentioned',
       'crm_customer_payment_intention', 'crm_competitors_mentioned',
       'crm_membership_level', 'crm_platform_issues_raised',
       'crm_agent_chased_contractor', 'crm_agent_chase_count',
       'crm_accreditation_issues', 'crm_membership_overdue',
       'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price',
       'crm_customer_complained', 'crm_refund_mentioned',
       'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
       'crm_financial_hardship_mentioned', 'year'],
      dtype='str')

Important Columns:

crm_customer_complained, crm_negative_customer_experience,crm_financial_hardship_mentioned ,crm_dissatisified_with_renewal_price, crm_customer_payment_intention, 

#### Renewal_calls

In [83]:
renewal_calls_clean['Customer_Response'].value_counts(dropna=False)

Customer_Response
NaN              68617
neutral          55968
not discussed    22315
negative          5652
positive           717
Name: count, dtype: int64

Insights:

neutral - most  
negative - meaningful  
positive - very few  
not discussed - noise  
NaN - missing

So:
Only "negative" = true negative signal  
neutral / not discussed = NOT negative

In [84]:
renewal_calls_clean['Desire_To_Cancel'].value_counts(dropna=False)

Desire_To_Cancel
NaN                                                                                                                     68620
renewed                                                                                                                 38612
not discussed                                                                                                           34207
desired to cancel                                                                                                       10434
renew                                                                                                                     679
                                                                                                                        ...  
desired to cancel (initially, but later agreed to proceed with the accreditation process)                                   1
desired to cancel (conditionally, pending client's contract renewal)                                 

Insights:

"desired to cancel" - churn signal  
"renewed" / "renew" - NOT churn  
"not discussed" - neutral  

In [85]:
renewal_calls_clean['Explicit_Switching_Intent'].value_counts(dropna=False) 

Explicit_Switching_Intent
no                                                                                                          84540
NaN                                                                                                         68633
yes                                                                                                            64
xxxx                                                                                                           26
unavailable                                                                                                     2
99) mentioned on the website, implying that alternative, potentially cheaper options might be available.        1
prompt 3: no                                                                                                    1
prompt 4:                                                                                                       1
[no]                                                          

Insights:

no - dominant  
yes - real signal  
others (xxxx, prompt...) - noise

So:

"yes" = 1
else =0 

In [86]:
renewal_calls_clean['Complaint_Category'].value_counts(dropna=False)

Complaint_Category
NaN                                135320
price increase / rise and value      3205
payment issues                       2772
accreditation and certification      2731
process and auto-renewal issues      1820
customer service                     1274
billings and invoices                1160
portal and technical issues          1138
quality and satisfaction              643
cancellation and refunds              641
account information                   567
audit                                 389
package and benefits                  310
financial hardship                    296
login and access                      255
discounts                             187
assessment                            180
sales and marketing                   132
others                                 64
compliance                             61
insurance                              56
privacy and security                   48
not mentioned                          20
Name: count, dt

Insight:

NaN → no complaint  
others → complaint exists

1. Clean Renewal Call Data

We standardize text and handle missing values to ensure consistent feature extraction.

In [87]:
cols = ['Customer_Response', 'Desire_To_Cancel', 'Explicit_Switching_Intent', 'Complaint_Category']

for col in cols:
    renewal_calls_clean[col] = renewal_calls_clean[col].fillna('unknown')
    renewal_calls_clean[col] = renewal_calls_clean[col].str.lower().str.strip()

2. Create Behavioral Features

We extract meaningful churn signals using pattern-based logic.

In [88]:
# Negative sentiment
renewal_calls_clean['is_negative'] = (
    renewal_calls_clean['Customer_Response'] == 'negative'
).astype(int)

# Complaint flag
renewal_calls_clean['has_complaint'] = (
    (renewal_calls_clean['Complaint_Category'] != 'unknown') &
    (renewal_calls_clean['Complaint_Category'] != 'not mentioned')
).astype(int)

# Cancel intent 
renewal_calls_clean['cancel_intent'] = (
    renewal_calls_clean['Desire_To_Cancel'].str.contains('cancel', na=False)
).astype(int)

# Switching intent
renewal_calls_clean['switch_intent'] = (
    renewal_calls_clean['Explicit_Switching_Intent'] == 'yes'
).astype(int)

3. Aggregate Customer Behavior

We summarize interaction behavior at the customer level.

In [89]:
renewal_features = renewal_calls_clean.groupby('Co_Ref').agg(
    negative_calls=('is_negative', 'sum'),
    complaint_calls=('has_complaint', 'sum'),
    cancel_intent_calls=('cancel_intent', 'sum'),
    switching_intent_calls=('switch_intent', 'sum')
).reset_index()

#### CC_calls

In [90]:
cc_calls_clean['cc_contractor_complained'].value_counts(dropna=False)


cc_contractor_complained
no                                                                                                                                                                                                                      29260
yes                                                                                                                                                                                                                      2219
NaN                                                                                                                                                                                                                        95
not applicable                                                                                                                                                                                                             50
the contractor was appreciative of the agent's efforts and requested an email to confir

In [91]:
cc_calls_clean['cc_business_struggles_financial_hardship'].value_counts(dropna=False)


cc_business_struggles_financial_hardship
no                                                                                                       30530
yes                                                                                                        939
NaN                                                                                                        136
[yes/no]                                                                                                    23
not applicable                                                                                               6
or a complete"" offering from scanlight."                                                                    1
implying that they have an external hr system that is compatible with safecontractor's requirements."        1
Name: count, dtype: int64

In [92]:
cc_calls_clean['cc_pricing_mentioned'].value_counts(dropna=False)


cc_pricing_mentioned
no               27586
yes               3899
NaN                 95
not discussed       21
90                  14
80                  10
20                   4
70                   2
50                   2
100                  1
95                   1
85                   1
Name: count, dtype: int64

In [93]:
cc_calls_clean['cc_platform_issues'].value_counts(dropna=False)


cc_platform_issues
no                                          29432
yes                                          2159
NaN                                            33
[yes/no]                                       11
which they believe is a breach of gdpr."        1
Name: count, dtype: int64

In [94]:
cc_calls_clean['cc_dissatisfaction_support'].value_counts(dropna=False)

cc_dissatisfaction_support
no                                                                                                                                                                                                         30907
yes                                                                                                                                                                                                          681
NaN                                                                                                                                                                                                           36
[yes/no]                                                                                                                                                                                                      11
the customer is frustrated with the repetitive nature of the questionnaire and the time it takes to complete it, mentioning that they usu

Observations from 5 cells:

Data is so messy

So:
Using pattern based cleaning

"yes" - 1
everything else - 0

1. Clean CC_calls Data

We standardize categorical values and handle inconsistent entries.

In [95]:
cols = [
    'cc_contractor_complained',
    'cc_business_struggles_financial_hardship',
    'cc_pricing_mentioned',
    'cc_platform_issues',
    'cc_dissatisfaction_support'
]

for col in cols:
    cc_calls_clean[col] = cc_calls_clean[col].fillna('unknown')
    cc_calls_clean[col] = cc_calls_clean[col].astype(str).str.lower().str.strip()

2. Create Customer Care Features

We convert messy categorical values into clean binary indicators.

In [96]:
# Complaint
cc_calls_clean['complaint_flag'] = (
    cc_calls_clean['cc_contractor_complained'].str.contains('yes', na=False)
).astype(int)

# Financial stress
cc_calls_clean['financial_stress'] = (
    cc_calls_clean['cc_business_struggles_financial_hardship'].str.contains('yes', na=False)
).astype(int)

# Pricing issue
cc_calls_clean['pricing_issue'] = (
    cc_calls_clean['cc_pricing_mentioned'].str.contains('yes', na=False)
).astype(int)

# Platform issue
cc_calls_clean['platform_issue'] = (
    cc_calls_clean['cc_platform_issues'].str.contains('yes', na=False)
).astype(int)

# Support dissatisfaction
cc_calls_clean['support_dissatisfaction'] = (
    cc_calls_clean['cc_dissatisfaction_support'].str.contains('yes', na=False)
).astype(int)

3. Aggregate Customer Care Behavior

We summarize issue-related behavior at customer level.

In [97]:
cc_features = cc_calls_clean.groupby('Co_Ref').agg(
    cc_complaint_calls=('complaint_flag', 'sum'),
    cc_financial_stress_calls=('financial_stress', 'sum'),
    cc_pricing_issue_calls=('pricing_issue', 'sum'),
    cc_platform_issue_calls=('platform_issue', 'sum'),
    cc_support_dissatisfaction_calls=('support_dissatisfaction', 'sum')
).reset_index()

#### emails

In [98]:
emails_clean['crm_customer_complained'].value_counts(dropna=False)


crm_customer_complained
no                                                                          104315
NaN                                                                          11475
yes                                                                           7568
not discussed                                                                   16
not applicable (there is no content to analyze)                                  2
not applicable (there is no call transcript or email content to analyze)         2
not applicable (there is no call mentioned in the email content)                 2
not applicable (there is no call transcript provided)                            2
[yes/no]                                                                         1
not applicable (no email content provided)                                       1
not applicable (there is no email content to analyze)                            1
not applicable (no conversation provided)                      

In [99]:
emails_clean['crm_negative_customer_experience'].value_counts(dropna=False)


crm_negative_customer_experience
no                                                       66587
not discussed                                            26962
yes                                                      18360
NaN                                                      11475
and also expressed concerns about the audit process          1
[yes/no/not discussed]                                       1
causing them to lose work."                                  1
not applicable (no email content provided)                   1
not applicable (there is no email content to analyze)        1
Name: count, dtype: int64

In [100]:
emails_clean['crm_financial_hardship_mentioned'].value_counts(dropna=False)


crm_financial_hardship_mentioned
not discussed                                                                                                                                   72242
no                                                                                                                                              33120
NaN                                                                                                                                             11475
yes                                                                                                                                              6545
city is required"" message."                                                                                                                        1
bye""."                                                                                                                                             1
[yes/no/not discussed]                                             

In [101]:
emails_clean['crm_dissatisified_with_renewal_price'].value_counts(dropna=False)


crm_dissatisified_with_renewal_price
not discussed                                                                 70611
no                                                                            33063
NaN                                                                           11155
yes                                                                            8519
0                                                                                38
2                                                                                 2
no actions or steps were taken by the agent in the provided email content.        1
Name: count, dtype: int64

In [102]:
emails_clean['crm_customer_payment_intention'].value_counts(dropna=False)

crm_customer_payment_intention
not discussed    69476
yes              28236
NaN              21035
no                4638
0                    2
20                   1
30                   1
Name: count, dtype: int64

Observations from 5 cells:

Pattern across all columns are 
yes / no / not discussed / NaN / random text

So:
"yes" = signal
everything else = 0 

1. Clean Email Data

We standardize text and handle missing values.

In [103]:
cols = [
    'crm_customer_complained',
    'crm_negative_customer_experience',
    'crm_financial_hardship_mentioned',
    'crm_dissatisified_with_renewal_price',
    'crm_customer_payment_intention'
]

for col in cols:
    emails_clean[col] = emails_clean[col].fillna('unknown')
    emails_clean[col] = emails_clean[col].astype(str).str.lower().str.strip()

2. Create Email Behavioral Features

We convert categorical values into binary signals:

- "yes" - 1 (true signal)
- everything else - 0

This ensures robust handling of messy text data.

In [104]:
# Complaint
emails_clean['email_complaint'] = (
    emails_clean['crm_customer_complained'].str.contains('yes', na=False)
).astype(int)

# Negative experience
emails_clean['email_negative_exp'] = (
    emails_clean['crm_negative_customer_experience'].str.contains('yes', na=False)
).astype(int)

# Financial hardship
emails_clean['email_financial_issue'] = (
    emails_clean['crm_financial_hardship_mentioned'].str.contains('yes', na=False)
).astype(int)

# Price dissatisfaction
emails_clean['email_price_issue'] = (
    emails_clean['crm_dissatisified_with_renewal_price'].str.contains('yes', na=False)
).astype(int)

# Payment intention (reverse signal!)
emails_clean['low_payment_intent'] = (
    emails_clean['crm_customer_payment_intention'].str.contains('no', na=False)
).astype(int)

4. Fix Numeric Column

The column crm_agent_chase_count is stored as string.

We convert it to numeric before aggregation to avoid errors.

In [105]:
emails_clean['crm_agent_chase_count'] = pd.to_numeric(
    emails_clean['crm_agent_chase_count'],
    errors='coerce'
)

emails_clean['crm_agent_chase_count'] = emails_clean['crm_agent_chase_count'].fillna(0)

4. Aggregate Email Features

We aggregate email-level interactions into customer-level features.

Each row becomes:
1 customer x summarized email behavior

In [106]:
email_features = emails_clean.groupby('Co_Ref').agg(
    email_complaints=('email_complaint', 'sum'),
    email_negative_exp=('email_negative_exp', 'sum'),
    email_financial_issue=('email_financial_issue', 'sum'),
    email_price_issue=('email_price_issue', 'sum'),
    low_payment_intent_count=('low_payment_intent', 'sum'),
    email_interactions=('crm_agent_chase_count', 'sum')
).reset_index()

 Merge All Behavioral Features

We merge:
- renewal features
- customer care features
- email features

into the main dataset.

In [107]:
df = df.merge(renewal_features, on='Co_Ref', how='left')


In [108]:
df = df.merge(cc_features, on='Co_Ref', how='left')


In [109]:
df = df.merge(email_features, on='Co_Ref', how='left')

In [110]:
df.shape

(113894, 32)

In [111]:
df.isnull().sum()

Co_Ref                                  0
total_interactions                      0
cc_call_ratio                           0
low_engagement                          0
total_emails                            0
tenure_bucket                           0
Band                                    0
Anchor_Group                            0
Payment_Method                          0
Payment_Timeframe                       0
Renewal_Year                            0
Sustainability_Score                    0
Total_Renewal_Score_New                 0
Auto_Renewal_Score                      0
Anchoring_Score                         0
Tenure_Scores                           0
churn                                   0
negative_calls                      20407
complaint_calls                     20407
cancel_intent_calls                 20407
switching_intent_calls              20407
cc_complaint_calls                  72673
cc_financial_stress_calls           72673
cc_pricing_issue_calls            

Handle Missing Values

After merging, some customers may not have:

- calls  
- emails  
- complaints  

These appear as missing values.

We replace them with 0, meaning:
"No interaction / No signal".

In [112]:
new_cols = [
    'negative_calls', 'complaint_calls', 'cancel_intent_calls', 'switching_intent_calls',
    'cc_complaint_calls', 'cc_financial_stress_calls', 'cc_pricing_issue_calls',
    'cc_platform_issue_calls', 'cc_support_dissatisfaction_calls',
    'email_complaints', 'email_negative_exp', 'email_financial_issue',
    'email_price_issue', 'low_payment_intent_count', 'email_interactions'
]

df[new_cols] = df[new_cols].fillna(0)

In [113]:
df.shape

(113894, 32)

In [114]:
df.isnull().sum()

Co_Ref                              0
total_interactions                  0
cc_call_ratio                       0
low_engagement                      0
total_emails                        0
tenure_bucket                       0
Band                                0
Anchor_Group                        0
Payment_Method                      0
Payment_Timeframe                   0
Renewal_Year                        0
Sustainability_Score                0
Total_Renewal_Score_New             0
Auto_Renewal_Score                  0
Anchoring_Score                     0
Tenure_Scores                       0
churn                               0
negative_calls                      0
complaint_calls                     0
cancel_intent_calls                 0
switching_intent_calls              0
cc_complaint_calls                  0
cc_financial_stress_calls           0
cc_pricing_issue_calls              0
cc_platform_issue_calls             0
cc_support_dissatisfaction_calls    0
email_compla

In [115]:
df.head()

,Co_Ref,total_interactions,cc_call_ratio,low_engagement,total_emails,tenure_bucket,Band,Anchor_Group,Payment_Method,Payment_Timeframe,...,cc_financial_stress_calls,cc_pricing_issue_calls,cc_platform_issue_calls,cc_support_dissatisfaction_calls,email_complaints,email_negative_exp,email_financial_issue,email_price_issue,low_payment_intent_count,email_interactions
0,vt6174,1.0,0.000000,0,1.0,1-3,band c1,1,card,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,vd3828,1.0,0.000000,0,1.0,0-1,band c1,1,card,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,2.0
2,dv8120,0.0,0.000000,1,0.0,5-10,band c1,1,card,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,2.0
3,ez9894,7.0,0.625000,0,2.0,10+,band c1,1,card,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,7.0
4,fa8957,6.0,0.285714,0,4.0,1-3,band c1,1,card,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,4.0,8.0


In [116]:
df.dtypes

Co_Ref                                   str
total_interactions                   float64
cc_call_ratio                        float64
low_engagement                         int64
total_emails                         float64
tenure_bucket                       category
Band                                category
Anchor_Group                        category
Payment_Method                      category
Payment_Timeframe                    float64
Renewal_Year                        category
Sustainability_Score                 float64
Total_Renewal_Score_New              float64
Auto_Renewal_Score                     int64
Anchoring_Score                      float64
Tenure_Scores                        float64
churn                                  int64
negative_calls                       float64
complaint_calls                      float64
cancel_intent_calls                  float64
switching_intent_calls               float64
cc_complaint_calls                   float64
cc_financi

In [117]:
df['tenure_bucket'] = 'TB_' + df['tenure_bucket'].astype(str)

In [119]:
df.to_csv('final_churn_dataset.csv', index=False)

In [120]:
df.head()

,Co_Ref,total_interactions,cc_call_ratio,low_engagement,total_emails,tenure_bucket,Band,Anchor_Group,Payment_Method,Payment_Timeframe,...,cc_financial_stress_calls,cc_pricing_issue_calls,cc_platform_issue_calls,cc_support_dissatisfaction_calls,email_complaints,email_negative_exp,email_financial_issue,email_price_issue,low_payment_intent_count,email_interactions
0,vt6174,1.0,0.000000,0,1.0,TB_1-3,band c1,1,card,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,vd3828,1.0,0.000000,0,1.0,TB_0-1,band c1,1,card,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,2.0
2,dv8120,0.0,0.000000,1,0.0,TB_5-10,band c1,1,card,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,2.0
3,ez9894,7.0,0.625000,0,2.0,TB_10+,band c1,1,card,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,7.0
4,fa8957,6.0,0.285714,0,4.0,TB_1-3,band c1,1,card,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,4.0,8.0


In [121]:
df['tenure_bucket'].unique()

<StringArray>
['TB_1-3', 'TB_0-1', 'TB_5-10', 'TB_10+', 'TB_3-5', 'TB_Unknown']
Length: 6, dtype: str